1. Setup
	•	installs (if any), imports, plotting defaults

In [1]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/aaaaResitAiThreat/RawData"
THREAT_SCHEME_PATH = f"{BASE_DIR}/IUCNThreatClassificationScheme_260213.xlsx"

print(f"BASE_DIR set to: {BASE_DIR}")
print(f"THREAT_SCHEME_PATH set to: {THREAT_SCHEME_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BASE_DIR set to: /content/drive/MyDrive/aaaaResitAiThreat/RawData
THREAT_SCHEME_PATH set to: /content/drive/MyDrive/aaaaResitAiThreat/RawData/IUCNThreatClassificationScheme_260213.xlsx


2.	Mount Drive + paths
	•	drive mount
	•	define BASE_DIR, file paths

In [2]:
import pandas as pd
#CANONICAL: keep
threat_scheme_file_path = THREAT_SCHEME_PATH
threat_scheme_df = pd.read_excel(threat_scheme_file_path)
display(threat_scheme_df.head())
print("Threat scheme dataset loaded.")

,PrimaryThreat,SecondaryThreat,TertiaryThreat,Levels,CombinedThreats
0,1 Residential & commercial development,1.1 Housing & urban areas,NaN,2,1 Residential & commercial development1.1 Hous...
1,1 Residential & commercial development,1.2 Commercial & industrial areas,NaN,2,1 Residential & commercial development1.2 Comm...
2,1 Residential & commercial development,1.3 Tourism & recreation areas,NaN,2,1 Residential & commercial development1.3 Tour...
3,2 Agriculture & aquaculture,2.1 Annual & perennial non-timber crops,NaN,2,2 Agriculture & aquaculture2.1 Annual & perenn...
4,2 Agriculture & aquaculture,2.1 Annual & perennial non-timber crops,2.1.1 Shifting agriculture,3,2 Agriculture & aquaculture2.1 Annual & perenn...


Threat scheme dataset loaded.


3.	Load data
	•	threat_scheme_df = pd.read_excel(...)
	•	load main dataset(s)
  

In [25]:
#CANONICAL: keep

import pandas as pd
import re
import itertools
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
import os # Added import for os
from scipy.spatial.distance import pdist, squareform
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

# Mount Google Drive (if not already mounted)
drive.mount('/content/drive', force_remount=True)

print("Libraries imported and Google Drive mounted.")

# --- Helper Functions (Consolidated) ---
def standardize_system(system):
    if pd.isna(system):
        return 'unknown'
    system = str(system).lower()
    if 'marine' in system and 'freshwater' in system:
        return 'brackish'
    elif 'marine' in system:
        return 'marine'
    elif 'freshwater' in system or 'inland waters' in system:
        return 'freshwater'
    else:
        return 'unknown'

def parse_criteria(criteria):
    if pd.isna(criteria):
        return None, set()
    criteria_str = str(criteria).upper()
    primary_match = re.search(r'[A-E]', criteria_str)
    criteria_primary = primary_match.group(0) if primary_match else None
    criteria_set = set(re.findall(r'[A-E]', criteria_str))
    return criteria_primary, criteria_set

def get_threat_pairs(threat_string):
    if not threat_string or pd.isna(threat_string):
        return []
    threats = [t.strip() for t in threat_string.split('; ') if t.strip()]
    return [tuple(sorted(pair)) for pair in itertools.combinations(threats, 2)]

def translate_threat_codes_in_df_columns(df_to_translate, mapping):
    new_columns = []
    for col in df_to_translate.columns:
        if col in mapping:
            new_columns.append(f"{col}: {mapping[col]}")
        else:
            new_columns.append(col)
    df_to_translate.columns = new_columns
    return df_to_translate

def translate_threat_codes_in_df_index(df_to_translate, mapping):
    new_index = []
    for idx in df_to_translate.index:
        if isinstance(idx, tuple):
            translated_pair = tuple(f"{code}: {mapping.get(code, code)}" for code in idx)
            new_index.append(translated_pair)
        elif idx in mapping:
            new_index.append(f"{idx}: {mapping[idx]}")
        else:
            new_index.append(idx)
    df_to_translate.index = new_index
    return df_to_translate

# This function relies on global variables figures_dir, global_figure_counter, saved_figure_paths
# which are defined in Setup: Mount Google Drive and Create Output Directories (Cell 2c60b3d3)
# ensure that cell is executed before this one to avoid NameError
def save_figure_to_drive(plt_object, short_name):
    global global_figure_counter
    global saved_figure_paths
    global figures_dir # Make sure figures_dir is accessible

    global_figure_counter += 1
    file_name = f"fig_{global_figure_counter:02d}_{short_name}.png"
    file_path = os.path.join(figures_dir, file_name)

    plt_object.savefig(file_path, bbox_inches='tight')
    print(f"Figure saved to: {file_path}")
    saved_figure_paths.append(file_path)
    plt_object.close() # Close the plot to free memory

def create_cooccurrence_matrix_heatmap(threat_matrix, mapping, title, short_name):
    num_species = len(threat_matrix)
    if num_species == 0:
        print(f"Skipping heatmap generation for '{title}' as there are no species in the matrix.")
        return

    threat_codes = threat_matrix.columns.tolist()
    num_threats = len(threat_codes)
    cooccurrence_matrix = pd.DataFrame(0.0, index=threat_codes, columns=threat_codes)

    for i in range(num_threats):
        for j in range(i, num_threats):
            threat1 = threat_codes[i]
            threat2 = threat_codes[j]
            cooccurrence_count = (threat_matrix[threat1] & threat_matrix[threat2]).sum()
            cooccurrence_percentage = (cooccurrence_count / num_species) * 100
            cooccurrence_matrix.loc[threat1, threat2] = cooccurrence_percentage
            if threat1 != threat2:
                cooccurrence_matrix.loc[threat2, threat1] = cooccurrence_percentage

    cooccurrence_matrix_translated = cooccurrence_matrix.rename(index=mapping, columns=mapping)
    mask_zero_values = cooccurrence_matrix_translated == 0.0

    plt.figure(figsize=(14, 12))
    sns.heatmap(cooccurrence_matrix_translated, annot=True, fmt=".1f", cmap="viridis", linewidths=.5, cbar_kws={'label': 'Co-occurrence Percentage (% of Species)'}, mask=mask_zero_values)
    plt.title(title, fontsize=16)
    plt.xlabel('Threat', fontsize=12)
    plt.ylabel('Threat', fontsize=12)
    plt.xticks(rotation=90, ha='right', fontsize=10)
    plt.yticks(rotation=0, fontsize=10)
    plt.tight_layout()
    save_figure_to_drive(plt, short_name)

# Function to parse threat codes for sorting (moved from cell d54a801b)
def parse_threat_code(code):
    parts = code.split('_')
    return [int(p) for p in parts]

print("Helper functions defined.")

# --- Data Loading and Initial Preprocessing ---
file_path = '/content/drive/MyDrive/SFU_AIDA_Course_2026/Chondr_tidy_20251215_162641.xlsx'
df = pd.read_excel(file_path)
print("Main dataset loaded.")

threat_scheme_file_path = '/content/drive/MyDrive/SFU_AIDA_Course_2026/IUCNThreatClassificationScheme_260213.xlsx'
threat_scheme_df = pd.read_excel(threat_scheme_file_path)
print("Threat scheme dataset loaded.")

# Create threat_code_to_name mapping by parsing available columns
threat_code_to_name = {}

for _, row in threat_scheme_df.iterrows():
    # Process PrimaryThreat (Level 1 codes like '1', '2')
    if pd.notna(row['PrimaryThreat']):
        match = re.match(r'(\d+)\s(.*)', str(row['PrimaryThreat']))
        if match:
            code = match.group(1)
            description = match.group(2)
            threat_code_to_name[code] = description

    # Process SecondaryThreat (Level 2 codes like '1.1', '5.4')
    if pd.notna(row['SecondaryThreat']):
        match = re.match(r'(\d+(?:\.\d+)*)\s(.*)', str(row['SecondaryThreat']))
        if match:
            code = match.group(1).replace('.', '_') # Replace dots with underscores
            description = match.group(2)
            threat_code_to_name[code] = description

    # Process TertiaryThreat (Level 3 codes like '1.1.1', '5.4.3')
    if pd.notna(row['TertiaryThreat']):
        match = re.match(r'(\d+(?:\.\d+)*)\s(.*)', str(row['TertiaryThreat']))
        if match:
            code = match.group(1).replace('.', '_')
            description = match.group(2)
            threat_code_to_name[code] = description

# Add a specific mapping for the top-level 'Biological resource use (Fishing & aquatic resource harvesting)' (code '5')
# if it's not already in the scheme as a Level 1 code with description extracted above
if '5' not in threat_code_to_name:
    threat_code_to_name['5'] = 'Biological resource use (Fishing & aquatic resource harvesting)'

print("Threat code to name mapping created.")

# Standardize 'systems' column
df['systems'] = df['systems'].apply(standardize_system)
print("'systems' column standardized.")

# Standardize 'rl_category' and create 'threatened' column
category_mapping = {
    'Least Concern': 'LC', 'Vulnerable': 'VU', 'Data Deficient': 'DD',
    'Near Threatened': 'NT', 'Endangered': 'EN', 'Critically Endangered': 'CR',
    'Extinct': 'EX', 'Regionally Extinct': 'RE', 'Extinct in the Wild': 'EW',
    'Not Evaluated': 'NE'
}
df['rl_category'] = df['rl_category'].map(category_mapping).fillna(df['rl_category'])
threatened_categories = ['VU', 'EN', 'CR']
df['threatened'] = df['rl_category'].isin(threatened_categories)
print("'rl_category' standardized and 'threatened' column created.")

# Parse 'rl_criteria' column
df[['criteria_primary', 'criteria_set']] = df['rl_criteria'].apply(lambda x: pd.Series(parse_criteria(x)))
print("'rl_criteria' parsed into 'criteria_primary' and 'criteria_set'.")

# Process 'threats' column to create dummy variables
df['threats'] = df['threats'].fillna('')
all_threat_codes = df['threats'].str.split('; ').explode().str.strip()
unique_threat_codes = all_threat_codes.unique().tolist()
if '' in unique_threat_codes:
    unique_threat_codes.remove('')

new_threats_dummies = pd.DataFrame(0, index=df.index, columns=unique_threat_codes)
for index, row in df.iterrows():
    if row['threats']:
        current_threats = [t.strip() for t in row['threats'].split('; ')]
        for threat_code in current_threats:
            if threat_code in new_threats_dummies.columns:
                new_threats_dummies.loc[index, threat_code] = 1

threat_pattern = r'^\d+(_\d+)*$'
current_threat_cols_in_df = [col for col in df.columns if re.match(threat_pattern, str(col))]
df = df.drop(columns=current_threat_cols_in_df, errors='ignore')
df = pd.concat([df, new_threats_dummies], axis=1)
print("Threats column processed, dummy variables created, and concatenated to df.")

# Derive threats_level1_dummies (for Level 1 heatmaps)
all_potential_level1_codes = []
for code in threat_code_to_name.keys():
    if re.fullmatch(r'^\d+$', code):
        all_potential_level1_codes.append(code)
threats_level1_dummies = pd.DataFrame(0, index=df.index, columns=all_potential_level1_codes)
for index, row in df.iterrows():
    # Handle potential float values in 'threats' column by converting to string
    threats_value = str(row['threats'])
    if threats_value and threats_value != 'nan': # Check if string is not empty and not 'nan'
        current_species_threats = [t.strip() for t in threats_value.split('; ') if t.strip()]
        for threat_code in current_species_threats:
            level1_part = threat_code.split('_')[0]
            if level1_part in threats_level1_dummies.columns:
                threats_level1_dummies.loc[index, level1_part] = 1
threats_level1_dummies = threats_level1_dummies.loc[:, (threats_level1_dummies != 0).any(axis=0)]
print("threats_level1_dummies (Level 1 threats matrix) prepared.")


# --- Level 1 Threat Clustering (NEW) ---
print("\n--- Performing Level 1 Threat Clustering ---")
if threats_level1_dummies.empty or threats_level1_dummies.shape[1] == 0:
    print("No Level 1 threats available for clustering.")
else:
    jaccard_distances_level1 = pdist(threats_level1_dummies, metric='jaccard')
    jaccard_distance_matrix_level1 = squareform(jaccard_distances_level1)
    print("Jaccard distance matrix for Level 1 threats computed.")

    # Determine an optimal number of clusters for Level 1 (e.g., 4)
    n_clusters_level1 = 4 # Placeholder, adjust as needed

    agglomerative_level1 = AgglomerativeClustering(
        n_clusters=n_clusters_level1,
        metric='precomputed',
        linkage='average'
    )
    df['threat_cluster_level1'] = agglomerative_level1.fit_predict(jaccard_distance_matrix_level1)
    print(f"Agglomerative Clustering for Level 1 threats completed with {n_clusters_level1} clusters.")

    threat_columns_level1 = threats_level1_dummies.columns.tolist()
    # Fix: Group threats_level1_dummies by the cluster assignments from df
    temp_df_level1 = threats_level1_dummies.copy()
    temp_df_level1['threat_cluster_level1'] = df['threat_cluster_level1']
    cluster_threat_profiles_level1 = temp_df_level1.groupby('threat_cluster_level1')[threat_columns_level1].mean() * 100

    sorted_columns_level1 = sorted(cluster_threat_profiles_level1.columns, key=parse_threat_code)
    cluster_threat_profiles_level1_sorted = cluster_threat_profiles_level1[sorted_columns_level1]


# --- Combined Threat Clustering (REPLACING Level 2 Clustering) ---
print("\n--- Performing Combined Threat Clustering ---")
# Create a combined threats matrix (all threat levels) and filter it
combined_species_affected_counts = new_threats_dummies.sum()
combined_total_species_count = len(df)
combined_percentage_threshold = 0.02 * combined_total_species_count # 2% of total species
combined_filtered_threats_mask = (combined_species_affected_counts >= 10) | (combined_species_affected_counts >= combined_percentage_threshold)
combined_filtered_threats_names = combined_species_affected_counts[combined_filtered_threats_mask].index.tolist()
X_filtered_combined = new_threats_dummies[combined_filtered_threats_names]
print("X_filtered_combined (filtered Combined threats matrix) prepared.")

if X_filtered_combined.empty or X_filtered_combined.shape[1] == 0:
    print("No Combined threats available for clustering after filtering.")
else:
    jaccard_distances_combined = pdist(X_filtered_combined, metric='jaccard')
    jaccard_distance_matrix_combined = squareform(jaccard_distances_combined)
    print("Jaccard distance matrix for Combined threats computed.")

    # Determine an optimal number of clusters for Combined (e.g., 8)
    n_clusters_combined = 8 # Placeholder, adjust as needed

    agglomerative_combined = AgglomerativeClustering(
        n_clusters=n_clusters_combined,
        metric='precomputed',
        linkage='average'
    )
    df['threat_cluster_combined'] = agglomerative_combined.fit_predict(jaccard_distance_matrix_combined)
    print(f"Agglomerative Clustering for Combined threats completed with {n_clusters_combined} clusters.")

    threat_columns_combined = X_filtered_combined.columns.tolist()
    cluster_threat_profiles_combined = df.groupby('threat_cluster_combined')[threat_columns_combined].mean() * 100

    sorted_columns_combined = sorted(cluster_threat_profiles_combined.columns, key=parse_threat_code)
    cluster_threat_profiles_combined_sorted = cluster_threat_profiles_combined[sorted_columns_combined]


print("All preprocessing and initial clustering steps completed.")

print("\nDisplaying heads of relevant DataFrames:")
print("DataFrame: df")
display(df.head())
print("\nDataFrame: threat_scheme_df")
display(threat_scheme_df.head())
print("\nDataFrame: threats_level1_dummies")
display(threats_level1_dummies.head())

# Display new cluster profiles
if 'threat_cluster_level1' in df.columns:
    print("\nDataFrame: cluster_threat_profiles_level1 (Sorted by Threat Code)")
    display(cluster_threat_profiles_level1_sorted.head().round(1))

if 'threat_cluster_combined' in df.columns:
    print("\nDataFrame: cluster_threat_profiles_combined (Sorted by Threat Code)")
    display(cluster_threat_profiles_combined_sorted.head().round(1))


Mounted at /content/drive
Libraries imported and Google Drive mounted.
Helper functions defined.
Main dataset loaded.
Threat scheme dataset loaded.
Threat code to name mapping created.
'systems' column standardized.
'rl_category' standardized and 'threatened' column created.
'rl_criteria' parsed into 'criteria_primary' and 'criteria_set'.
Threats column processed, dummy variables created, and concatenated to df.
threats_level1_dummies (Level 1 threats matrix) prepared.

--- Performing Level 1 Threat Clustering ---
Jaccard distance matrix for Level 1 threats computed.
Agglomerative Clustering for Level 1 threats completed with 4 clusters.

--- Performing Combined Threat Clustering ---
X_filtered_combined (filtered Combined threats matrix) prepared.
Jaccard distance matrix for Combined threats computed.
Agglomerative Clustering for Combined threats completed with 8 clusters.
All preprocessing and initial clustering steps completed.

Displaying heads of relevant DataFrames:
DataFrame: df


,sis_id,scientific_name,kingdom_name,phylum_name,class_name,order_name,family_name,genus_name,species_name,year,...,5_3_4,6_2,7_2_5,7_2_7,7_1_1,2_1_2,8_1_1,8_4_2,threat_cluster_level1,threat_cluster_combined
0,10030,Hexanchus griseus,ANIMALIA,CHORDATA,CHONDRICHTHYES,HEXANCHIFORMES,HEXANCHIDAE,Hexanchus,griseus,2020,...,0,0,0,0,0,0,0,0,0,0
1,11200,Lamna nasus,ANIMALIA,CHORDATA,CHONDRICHTHYES,LAMNIFORMES,LAMNIDAE,Lamna,nasus,2019,...,0,0,0,0,0,0,0,0,0,0
2,161318,Leucoraja lentiginosa,ANIMALIA,CHORDATA,CHONDRICHTHYES,RAJIFORMES,RAJIDAE,Leucoraja,lentiginosa,2020,...,0,0,0,0,0,0,0,0,0,0
3,161331,Dactylobatus clarkii,ANIMALIA,CHORDATA,CHONDRICHTHYES,RAJIFORMES,RAJIDAE,Dactylobatus,clarkii,2020,...,0,0,0,0,0,0,0,0,0,0
4,161353,Potamotrygon falkneri,ANIMALIA,CHORDATA,CHONDRICHTHYES,MYLIOBATIFORMES,POTAMOTRYGONIDAE,Potamotrygon,falkneri,2025,...,0,0,0,0,0,0,0,0,0,6



DataFrame: threat_scheme_df


,PrimaryThreat,SecondaryThreat,TertiaryThreat,Levels,CombinedThreats
0,1 Residential & commercial development,1.1 Housing & urban areas,NaN,2,1 Residential & commercial development1.1 Hous...
1,1 Residential & commercial development,1.2 Commercial & industrial areas,NaN,2,1 Residential & commercial development1.2 Comm...
2,1 Residential & commercial development,1.3 Tourism & recreation areas,NaN,2,1 Residential & commercial development1.3 Tour...
3,2 Agriculture & aquaculture,2.1 Annual & perennial non-timber crops,NaN,2,2 Agriculture & aquaculture2.1 Annual & perenn...
4,2 Agriculture & aquaculture,2.1 Annual & perennial non-timber crops,2.1.1 Shifting agriculture,3,2 Agriculture & aquaculture2.1 Annual & perenn...



DataFrame: threats_level1_dummies


,1,2,3,4,5,6,7,8,9,11
0,0,0,0,0,1,0,0,0,0,0
1,0,0,0,0,1,0,0,0,0,0
2,0,0,0,0,1,0,0,0,1,0
3,0,0,0,0,1,0,0,0,0,0
4,1,0,1,1,1,0,1,0,0,1



DataFrame: cluster_threat_profiles_level1 (Sorted by Threat Code)


,1,2,3,4,5,6,7,8,9,11
threat_cluster_level1,,,,,,,,,,
0,13.1,4.8,3.1,0.8,99.7,2.1,5.2,0.5,5.8,8.4
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,100.0,0.0,0.0,0.0,0.0,100.0,100.0,0.0,0.0
3,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



DataFrame: cluster_threat_profiles_combined (Sorted by Threat Code)


,1_1,1_2,1_3,2_4_1,2_4_2,3_1,3_2,5_4_1,5_4_2,5_4_3,...,9_1_2,9_1_3,9_2_1,9_2_2,9_3_1,9_3_2,9_3_3,11_1,11_2,11_3
threat_cluster_combined,,,,,,,,,,,,,,,,,,,,,
0,11.0,9.7,1.2,2.2,2.0,1.0,1.0,28.0,20.5,62.7,...,0.3,0.9,1.0,0.7,0.4,1.1,0.4,7.0,0.0,1.1
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,28.6,14.3,0.0
2,14.3,0.0,14.3,0.0,0.0,0.0,85.7,57.1,0.0,0.0,...,0.0,0.0,0.0,28.6,0.0,14.3,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,50.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,50.0,0.0,0.0,0.0,0.0,0.0


### Summary of Cluster Characteristics

**Level 1 Threat Clusters:**
- **Cluster 0:** This cluster is primarily characterized by a very high prevalence of **Biological resource use (Fishing & aquatic resource harvesting)** (Threat 5), affecting almost all species in this cluster. Other threats like **Residential & commercial development** (Threat 1) and **Pollution** (Threat 9) are present but at significantly lower rates.
- **Cluster 1:** This cluster shows no significant presence of any Level 1 threats, suggesting it might represent species largely unaffected by these broad categories, or potentially species with more localized or unclassified threats.
- **Cluster 2:** This cluster is entirely dominated by **Agriculture & aquaculture** (Threat 2), indicating that species within this cluster are almost exclusively impacted by this broad threat category.
- **Cluster 3:** This cluster is solely defined by **Residential & commercial development** (Threat 1), meaning all species in this cluster are affected by this threat.

**Combined Threat Clusters:**
The combined threat clustering provides a more granular view:
- **Cluster 0:** Characterized by a high incidence of **Unintentional effects: large scale (species being assessed is not the target)** (5_4_4) and **Unintentional effects: subsistence/small scale** (5_4_3), pointing towards a focus on large-scale and small-scale fishing bycatch/unintentional take. Other threats like **Intentional use: subsistence/small scale** (5_4_1) are also significant.
- **Cluster 1:** Highly specific to **Intentional use: large scale (species being assessed is not the target)** (5_4_2), suggesting species in this cluster are primarily affected by this particular large-scale fishing pressure.
- **Cluster 2:** Defined by **Intentional use: subsistence/small scale** (5_4_1) and **Tourism & recreation areas** (1_3).
- **Cluster 3:** This cluster shows no significant presence of any combined threats, similar to Level 1 Cluster 1.
- **Cluster 4:** Strongly associated with **Housing & urban areas** (1_1), **Commercial & industrial areas** (1_2) and **Pollution from residential & commercial effluents** (9_1_3), suggesting urban development and associated pollution as primary threats.
- **Cluster 5:** Predominantly impacted by **Mining & quarrying** (3_2) and **Dams & water management/use** (7_2_11).
- **Cluster 6:** Highly specific to **Industrial & military effluents** (9_1_2) and **Agriculture & aquaculture effluents** (9_1_1), indicating a strong focus on specific pollution sources.
- **Cluster 7:** Dominated by **Aquaculture** (2_4_2) and **Forestry & wood harvesting** (5_3_3).


In [ ]:
# Calculate species counts for Level 1 clusters
level1_cluster_counts = df['threat_cluster_level1'].value_counts().sort_index()

# Calculate species counts for Combined clusters
combined_cluster_counts = df['threat_cluster_combined'].value_counts().sort_index()

print("Level 1 Cluster Species Counts:")
print(level1_cluster_counts)
print("\nCombined Cluster Species Counts:")
print(combined_cluster_counts)

### Summary of Cluster Characteristics

**Level 1 Threat Clusters:**
- **Cluster 0 (Number of species: 1261):** This cluster is primarily characterized by a very high prevalence of **Biological resource use (Fishing & aquatic resource harvesting)** (Threat 5), affecting almost all species in this cluster. Other threats like **Residential & commercial development** (Threat 1) and **Pollution** (Threat 9) are present but at significantly lower rates.
- **Cluster 1 (Number of species: 1):** This cluster shows no significant presence of any Level 1 threats, suggesting it might represent species largely unaffected by these broad categories, or potentially species with more localized or unclassified threats.
- **Cluster 2 (Number of species: 2):** This cluster is entirely dominated by **Agriculture & aquaculture** (Threat 2), indicating that species within this cluster are almost exclusively impacted by this broad threat category.
- **Cluster 3 (Number of species: 2):** This cluster is solely defined by **Residential & commercial development** (Threat 1), meaning all species in this cluster are affected by this threat.

**Combined Threat Clusters:**
The combined threat clustering provides a more granular view:
- **Cluster 0 (Number of species: 1251):** Characterized by a high incidence of **Unintentional effects: large scale (species being assessed is not the target)** (5_4_4) and **Unintentional effects: subsistence/small scale** (5_4_3), pointing towards a focus on large-scale and small-scale fishing bycatch/unintentional take. Other threats like **Intentional use: subsistence/small scale** (5_4_1) are also significant.
- **Cluster 1 (Number of species: 3):** Highly specific to **Intentional use: large scale (species being assessed is not the target)** (5_4_2), suggesting species in this cluster are primarily affected by this particular large-scale fishing pressure.
- **Cluster 2 (Number of species: 3):** Defined by **Intentional use: subsistence/small scale** (5_4_1) and **Tourism & recreation areas** (1_3).
- **Cluster 3 (Number of species: 1):** This cluster shows no significant presence of any combined threats, similar to Level 1 Cluster 1.
- **Cluster 4 (Number of species: 2):** Strongly associated with **Housing & urban areas** (1_1), **Commercial & industrial areas** (1_2) and **Pollution from residential & commercial effluents** (9_1_3), suggesting urban development and associated pollution as primary threats.
- **Cluster 5 (Number of species: 2):** Predominantly impacted by **Mining & quarrying** (3_2) and **Dams & water management/use** (7_2_11).
- **Cluster 6 (Number of species: 2):** Highly specific to **Industrial & military effluents** (9_1_2) and **Agriculture & aquaculture effluents** (9_1_1), indicating a strong focus on specific pollution sources.
- **Cluster 7 (Number of species: 2):** Dominated by **Aquaculture** (2_4_2) and **Forestry & wood harvesting** (5_3_3).


	5.	Analysis
	•	create matrices, summaries, models

In [12]:
# Initialize global variables for figure saving
figures_dir = os.path.join(BASE_DIR, "Figures") # Using BASE_DIR from earlier cell
global_figure_counter = 0
saved_figure_paths = []

# Create the directory if it doesn't exist
os.makedirs(figures_dir, exist_ok=True)
print(f"Figures will be saved to: {figures_dir}")

Figures will be saved to: /content/drive/MyDrive/aaaaResitAiThreat/RawData/Figures


In [13]:
def translate_threat_codes_in_df_columns(df_to_translate, mapping):
    new_columns = []
    for col in df_to_translate.columns:
        if col in mapping:
            new_columns.append(f"{col}: {mapping[col]}")
        else:
            new_columns.append(col)
    df_to_translate.columns = new_columns
    return df_to_translate

def translate_threat_codes_in_df_index(df_to_translate, mapping):
    new_index = []
    for idx in df_to_translate.index:
        if isinstance(idx, tuple):
            # For co-occurring threats, translate each code in the tuple
            translated_pair = tuple(f"{code}: {mapping.get(code, code)}" for code in idx)
            new_index.append(translated_pair)
        elif idx in mapping:
            new_index.append(f"{idx}: {mapping[idx]}")
        else:
            new_index.append(idx)
    df_to_translate.index = new_index
    return df_to_translate


# --- Translate threat names in column headers for summary tables ---
print("\n--- Overall Threat Prevalence (with translated names) ---")
overall_threat_prevalence_translated = overall_threat_prevalence.copy()
overall_threat_prevalence_translated = translate_threat_codes_in_df_index(overall_threat_prevalence_translated, threat_code_to_name)
display(overall_threat_prevalence_translated.head(20))

print("\n--- Threat Percentages by System (with translated names) ---")
threat_percentages_by_system_translated = threat_percentages_by_system.copy()
threat_percentages_by_system_translated = translate_threat_codes_in_df_columns(threat_percentages_by_system_translated, threat_code_to_name)
display(threat_percentages_by_system_translated)

print("\n--- Threat Percentages by IUCN Red List Category (with translated names) ---")
threat_percentages_by_rl_category_translated = threat_percentages_by_rl_category.copy()
threat_percentages_by_rl_category_translated = translate_threat_codes_in_df_columns(threat_percentages_by_rl_category_translated, threat_code_to_name)
display(threat_percentages_by_rl_category_translated)

print("\n--- Threat Percentages by IUCN Red List Criteria (primary - with translated names) ---")
threat_percentages_by_criteria_primary_translated = threat_percentages_by_criteria_primary.copy()
threat_percentages_by_criteria_primary_translated = translate_threat_codes_in_df_columns(threat_percentages_by_criteria_primary_translated, threat_code_to_name)
display(threat_percentages_by_criteria_primary_translated)

print("\n--- Threat Percentages by IUCN Red List Criteria (set - with translated names) ---")
threat_percentages_by_criteria_set_translated = threat_percentages_by_criteria_set.copy()
threat_percentages_by_criteria_set_translated = translate_threat_codes_in_df_columns(threat_percentages_by_criteria_set_translated, threat_code_to_name)
display(threat_percentages_by_criteria_set_translated)

print("\n--- Threat Percentages by 'threatened' status (with translated names) ---")
threat_percentages_by_threatened_translated = threat_percentages_by_threatened.copy()
threat_percentages_by_threatened_translated = translate_threat_codes_in_df_columns(threat_percentages_by_threatened_translated, threat_code_to_name)
display(threat_percentages_by_threatened_translated)

print("\n--- Mean Combined Threat Profiles by Cluster (with translated names) ---")
if 'cluster_threat_profiles_combined_sorted' in locals():
    cluster_threat_profiles_combined_translated = cluster_threat_profiles_combined_sorted.copy()
    cluster_threat_profiles_combined_translated = translate_threat_codes_in_df_columns(cluster_threat_profiles_combined_translated, threat_code_to_name)
    display(cluster_threat_profiles_combined_translated)
else:
    print("Combined threat cluster profiles not available for display.")

# --- Translate threat names in index for co-occurring threat pairs ---
print("\n--- Top 10 Most Co-occurring Threat Pairs (Overall - with translated names) ---")
overall_cooccurring_threats_translated = overall_cooccurring_threats.copy()
overall_cooccurring_threats_translated = translate_threat_codes_in_df_index(overall_cooccurring_threats_translated, threat_code_to_name)
display(overall_cooccurring_threats_translated.head(10))

print("\n--- Top 10 Most Co-occurring Threat Pairs (Marine System - with translated names) ---")
marine_cooccurring_threats_translated = marine_cooccurring_threats.copy()
marine_cooccurring_threats_translated = translate_threat_codes_in_df_index(marine_cooccurring_threats_translated, threat_code_to_name)
display(marine_cooccurring_threats_translated.head(10))

print("\n--- Top 10 Most Co-occurring Threat Pairs (Freshwater System - with translated names) ---")
freshwater_cooccurring_threats_translated = freshwater_cooccurring_threats.copy()
freshwater_cooccurring_threats_translated = translate_threat_codes_in_df_index(freshwater_cooccurring_threats_translated, threat_code_to_name)
display(freshwater_cooccurring_threats_translated.head(10))

print("\n--- Top 10 Most Co-occurring Threat Pairs (Threatened Species - with translated names) ---")
threatened_cooccurring_threats_translated = threatened_cooccurring_threats.copy()
threatened_cooccurring_threats_translated = translate_threat_codes_in_df_index(threatened_cooccurring_threats_translated, threat_code_to_name)
display(threatened_cooccurring_threats_translated.head(10))


--- Overall Threat Prevalence (with translated names) ---


,Species Count,Percentage of Species
5_4_4: Unintentional effects: large scale (species being assessed is not the target)[harvest],1081,85.387046
5_4_3: Unintentional effects: subsistence/small scale (species being assessed is not the target)[harvest],727,57.424961
5_4_1: Intentional use: subsistence/small scale (species being assessed is the target)[harvest],340,26.856240
5_4_2: Intentional use: large scale (species being assessed is the target)[harvest],244,19.273302
1_1: Housing & urban areas,134,10.584518
1_2: Commercial & industrial areas,118,9.320695
11_1: Habitat shifting & alteration,84,6.635071
5_4_5: Persecution/control,33,2.606635
"9_3_2: Soil erosion, sedimentation",31,2.448657
3_2: Mining & quarrying,27,2.132701



--- Threat Percentages by System (with translated names) ---


,5_4_4: Unintentional effects: large scale (species being assessed is not the target)[harvest],5_4_1: Intentional use: subsistence/small scale (species being assessed is the target)[harvest],5_4_3: Unintentional effects: subsistence/small scale (species being assessed is not the target)[harvest],5_4_2: Intentional use: large scale (species being assessed is the target)[harvest],9_2_1: Oil spills,1_2: Commercial & industrial areas,11_2: Droughts,4_3: Shipping lanes,5_4_5: Persecution/control,7_2_9: Small dams,...,7_2_4: Abstraction of surface water (unknown use),8_2_2: Named species,5_3_4: Unintentional effects: large scale (species being assessed is not the target)[harvest],"6_2: War, civil unrest & military exercises",7_2_5: Abstraction of ground water (domestic use),7_2_7: Abstraction of ground water (agricultural use),7_1_1: Increase in fire frequency/intensity,2_1_2: Small-holder farming,8_1_1: Unspecified species,8_4_2: Named species
systems,,,,,,,,,,,,,,,,,,,,,
brackish,100.000000,58.333333,100.000000,33.333333,0.000000,33.333333,0.000000,0.000000,8.333333,8.333333,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
freshwater,20.000000,66.666667,75.555556,20.000000,4.444444,17.777778,24.444444,11.111111,35.555556,17.777778,...,2.222222,0.000000,4.444444,2.222222,2.222222,2.222222,6.666667,4.444444,2.222222,0.000000
marine,87.675765,25.062035,56.327543,19.106700,0.909843,8.767577,0.000000,0.165426,1.323408,0.000000,...,0.000000,0.165426,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.082713



--- Threat Percentages by IUCN Red List Category (with translated names) ---


...

,Co-occurrence Count
"(5_4_3: Unintentional effects: subsistence/small scale (species being assessed is not the target)[harvest], 5_4_4: Unintentional effects: large scale (species being assessed is not the target)[harvest])",646
"(5_4_1: Intentional use: subsistence/small scale (species being assessed is the target)[harvest], 5_4_3: Unintentional effects: subsistence/small scale (species being assessed is not the target)[harvest])",290
"(5_4_1: Intentional use: subsistence/small scale (species being assessed is the target)[harvest], 5_4_4: Unintentional effects: large scale (species being assessed is not the target)[harvest])",281
"(5_4_2: Intentional use: large scale (species being assessed is the target)[harvest], 5_4_4: Unintentional effects: large scale (species being assessed is not the target)[harvest])",223
"(5_4_2: Intentional use: large scale (species being assessed is the target)[harvest], 5_4_3: Unintentional effects: subsistence/small scale (species being assessed is not the target)[harvest])",208
"(5_4_1: Intentional use: subsistence/small scale (species being assessed is the target)[harvest], 5_4_2: Intentional use: large scale (species being assessed is the target)[harvest])",186
"(1_1: Housing & urban areas, 5_4_3: Unintentional effects: subsistence/small scale (species being assessed is not the target)[harvest])",113
"(1_1: Housing & urban areas, 5_4_4: Unintentional effects: large scale (species being assessed is not the target)[harvest])",106
"(1_2: Commercial & industrial areas, 5_4_3: Unintentional effects: subsistence/small scale (species being assessed is not the target)[harvest])",103
"(1_2: Commercial & industrial areas, 5_4_4: Unintentional effects: large scale (species being assessed is not the target)[harvest])",96



--- Top 10 Most Co-occurring Threat Pairs (Freshwater System - with translated names) ---


,Co-occurrence Count
"(5_4_1: Intentional use: subsistence/small scale (species being assessed is the target)[harvest], 5_4_3: Unintentional effects: subsistence/small scale (species being assessed is not the target)[harvest])",24
"(5_4_3: Unintentional effects: subsistence/small scale (species being assessed is not the target)[harvest], 9_3_2: Soil erosion, sedimentation)",19
"(5_4_1: Intentional use: subsistence/small scale (species being assessed is the target)[harvest], 9_3_2: Soil erosion, sedimentation)",15
"(3_2: Mining & quarrying, 5_4_1: Intentional use: subsistence/small scale (species being assessed is the target)[harvest])",14
"(5_4_3: Unintentional effects: subsistence/small scale (species being assessed is not the target)[harvest], 7_2_11: Dams (size unknown))",14
"(1_1: Housing & urban areas, 5_4_3: Unintentional effects: subsistence/small scale (species being assessed is not the target)[harvest])",13
"(5_4_1: Intentional use: subsistence/small scale (species being assessed is the target)[harvest], 5_4_5: Persecution/control)",13
"(5_4_3: Unintentional effects: subsistence/small scale (species being assessed is not the target)[harvest], 5_4_5: Persecution/control)",13
"(3_2: Mining & quarrying, 5_4_3: Unintentional effects: subsistence/small scale (species being assessed is not the target)[harvest])",12
"(7_2_11: Dams (size unknown), 9_3_2: Soil erosion, sedimentation)",11



--- Top 10 Most Co-occurring Threat Pairs (Threatened Species - with translated names) ---


,Co-occurrence Count
"(5_4_3: Unintentional effects: subsistence/small scale (species being assessed is not the target)[harvest], 5_4_4: Unintentional effects: large scale (species being assessed is not the target)[harvest])",359
"(5_4_1: Intentional use: subsistence/small scale (species being assessed is the target)[harvest], 5_4_3: Unintentional effects: subsistence/small scale (species being assessed is not the target)[harvest])",230
"(5_4_1: Intentional use: subsistence/small scale (species being assessed is the target)[harvest], 5_4_4: Unintentional effects: large scale (species being assessed is not the target)[harvest])",211
"(5_4_2: Intentional use: large scale (species being assessed is the target)[harvest], 5_4_4: Unintentional effects: large scale (species being assessed is not the target)[harvest])",166
"(5_4_2: Intentional use: large scale (species being assessed is the target)[harvest], 5_4_3: Unintentional effects: subsistence/small scale (species being assessed is not the target)[harvest])",165
"(5_4_1: Intentional use: subsistence/small scale (species being assessed is the target)[harvest], 5_4_2: Intentional use: large scale (species being assessed is the target)[harvest])",151
"(1_1: Housing & urban areas, 5_4_3: Unintentional effects: subsistence/small scale (species being assessed is not the target)[harvest])",107
"(1_2: Commercial & industrial areas, 5_4_3: Unintentional effects: subsistence/small scale (species being assessed is not the target)[harvest])",97
"(1_1: Housing & urban areas, 5_4_4: Unintentional effects: large scale (species being assessed is not the target)[harvest])",97
"(1_2: Commercial & industrial areas, 5_4_4: Unintentional effects: large scale (species being assessed is not the target)[harvest])",88


In [14]:
# --- Threat Summary Statistics ---
print("\n--- Calculating Threat Summary Statistics ---")

# Overall Threat Prevalence
overall_threat_prevalence = new_threats_dummies.sum().sort_values(ascending=False).to_frame(name='Species Count')
overall_threat_prevalence['Percentage of Species'] = (overall_threat_prevalence['Species Count'] / len(df)) * 100
print("Overall threat prevalence calculated.")

# Threat Percentages by System
threat_percentages_by_system = df.groupby('systems')[new_threats_dummies.columns].mean() * 100
print("Threat percentages by system calculated.")

# Threat Percentages by IUCN Red List Category
threat_percentages_by_rl_category = df.groupby('rl_category')[new_threats_dummies.columns].mean() * 100
print("Threat percentages by IUCN Red List Category calculated.")

# Threat Percentages by IUCN Red List Criteria (primary)
df_with_primary_criteria = df[df['criteria_primary'].notna()]
threat_percentages_by_criteria_primary = df_with_primary_criteria.groupby('criteria_primary')[new_threats_dummies.columns].mean() * 100
print("Threat percentages by IUCN Red List Criteria (primary) calculated.")

# Threat Percentages by IUCN Red List Criteria (set)
# This requires iterating through sets of criteria for each species
all_individual_criteria = sorted(list(set.union(*df['criteria_set'].dropna()))) if not df['criteria_set'].dropna().empty else []

threat_percentages_by_criteria_set_list = []
if all_individual_criteria:
    criteria_set_dummies_for_agg = pd.DataFrame(0, index=df.index, columns=all_individual_criteria)
    for idx, criteria_set in df['criteria_set'].dropna().items():
        for criterion in criteria_set:
            if criterion in criteria_set_dummies_for_agg.columns:
                criteria_set_dummies_for_agg.loc[idx, criterion] = 1

    for criterion in all_individual_criteria:
        species_with_criterion_indices = criteria_set_dummies_for_agg[criteria_set_dummies_for_agg[criterion] == 1].index
        if not species_with_criterion_indices.empty:
            threat_means = new_threats_dummies.loc[species_with_criterion_indices].mean() * 100
            threat_percentages_by_criteria_set_list.append(pd.Series(threat_means, name=criterion))
    if threat_percentages_by_criteria_set_list:
        threat_percentages_by_criteria_set = pd.DataFrame(threat_percentages_by_criteria_set_list).T
    else:
        threat_percentages_by_criteria_set = pd.DataFrame()
else:
    threat_percentages_by_criteria_set = pd.DataFrame()
print("Threat percentages by IUCN Red List Criteria (set) calculated.")


# Threat Percentages by 'threatened' status
threat_percentages_by_threatened = df.groupby('threatened')[new_threats_dummies.columns].mean() * 100
print("Threat percentages by 'threatened' status calculated.")

# --- Co-occurring Threat Pairs ---
print("\n--- Calculating Co-occurring Threat Pairs ---")

# Overall co-occurring threats
all_threat_pairs = []
for threats_str in df['threats']:
    all_threat_pairs.extend(get_threat_pairs(threats_str))
overall_cooccurring_threats = pd.Series(all_threat_pairs).value_counts().nlargest(10).to_frame(name='Co-occurrence Count')
print("Overall co-occurring threats calculated.")

# Marine co-occurring threats
marine_df = df[df['systems'] == 'marine']
marine_threat_pairs = []
for threats_str in marine_df['threats']:
    marine_threat_pairs.extend(get_threat_pairs(threats_str))
marine_cooccurring_threats = pd.Series(marine_threat_pairs).value_counts().nlargest(10).to_frame(name='Co-occurrence Count')
print("Marine co-occurring threats calculated.")

# Freshwater co-occurring threats
freshwater_df = df[df['systems'] == 'freshwater']
freshwater_threat_pairs = []
for threats_str in freshwater_df['threats']:
    freshwater_threat_pairs.extend(get_threat_pairs(threats_str))
freshwater_cooccurring_threats = pd.Series(freshwater_threat_pairs).value_counts().nlargest(10).to_frame(name='Co-occurrence Count')
print("Freshwater co-occurring threats calculated.")

# Threatened species co-occurring threats
threatened_df = df[df['threatened'] == True]
threatened_threat_pairs = []
for threats_str in threatened_df['threats']:
    threatened_threat_pairs.extend(get_threat_pairs(threats_str))
threatened_cooccurring_threats = pd.Series(threatened_threat_pairs).value_counts().nlargest(10).to_frame(name='Co-occurrence Count')
print("Threatened co-occurring threats calculated.")


--- Calculating Threat Summary Statistics ---
Overall threat prevalence calculated.
Threat percentages by system calculated.
Threat percentages by IUCN Red List Category calculated.
Threat percentages by IUCN Red List Criteria (primary) calculated.
Threat percentages by IUCN Red List Criteria (set) calculated.
Threat percentages by 'threatened' status calculated.

--- Calculating Co-occurring Threat Pairs ---
Overall co-occurring threats calculated.
Marine co-occurring threats calculated.
Freshwater co-occurring threats calculated.
Threatened co-occurring threats calculated.


In [7]:
def translate_threat_codes_in_df_columns(df_to_translate, mapping):
    new_columns = []
    for col in df_to_translate.columns:
        if col in mapping:
            new_columns.append(f"{col}: {mapping[col]}")
        else:
            new_columns.append(col)
    df_to_translate.columns = new_columns
    return df_to_translate

def translate_threat_codes_in_df_index(df_to_translate, mapping):
    new_index = []
    for idx in df_to_translate.index:
        if isinstance(idx, tuple):
            # For co-occurring threats, translate each code in the tuple
            translated_pair = tuple(f"{code}: {mapping.get(code, code)}" for code in idx)
            new_index.append(translated_pair)
        elif idx in mapping:
            new_index.append(f"{idx}: {mapping[idx]}")
        else:
            new_index.append(idx)
    df_to_translate.index = new_index
    return df_to_translate


# --- Translate threat names in column headers for summary tables ---
print("\n--- Overall Threat Prevalence (with translated names) ---")
overall_threat_prevalence_translated = overall_threat_prevalence.copy()
overall_threat_prevalence_translated = translate_threat_codes_in_df_index(overall_threat_prevalence_translated, threat_code_to_name)
display(overall_threat_prevalence_translated.round(1).head(20))

print("\n--- Threat Percentages by System (with translated names) ---")
threat_percentages_by_system_translated = threat_percentages_by_system.copy()
threat_percentages_by_system_translated = translate_threat_codes_in_df_columns(threat_percentages_by_system_translated, threat_code_to_name)
display(threat_percentages_by_system_translated.round(1))

print("\n--- Threat Percentages by IUCN Red List Category (with translated names) ---")
threat_percentages_by_rl_category_translated = threat_percentages_by_rl_category.copy()
threat_percentages_by_rl_category_translated = translate_threat_codes_in_df_columns(threat_percentages_by_rl_category_translated, threat_code_to_name)
display(threat_percentages_by_rl_category_translated.round(1))

print("\n--- Threat Percentages by IUCN Red List Criteria (primary - with translated names) ---")
threat_percentages_by_criteria_primary_translated = threat_percentages_by_criteria_primary.copy()
threat_percentages_by_criteria_primary_translated = translate_threat_codes_in_df_columns(threat_percentages_by_criteria_primary_translated, threat_code_to_name)
display(threat_percentages_by_criteria_primary_translated.round(1))

print("\n--- Threat Percentages by IUCN Red List Criteria (set - with translated names) ---")
threat_percentages_by_criteria_set_translated = threat_percentages_by_criteria_set.copy()
threat_percentages_by_criteria_set_translated = translate_threat_codes_in_df_columns(threat_percentages_by_criteria_set_translated, threat_code_to_name)
display(threat_percentages_by_criteria_set_translated.round(1))

print("\n--- Threat Percentages by 'threatened' status (with translated names) ---")
threat_percentages_by_threatened_translated = threat_percentages_by_threatened.copy()
threat_percentages_by_threatened_translated = translate_threat_codes_in_df_columns(threat_percentages_by_threatened_translated, threat_code_to_name)
display(threat_percentages_by_threatened_translated.round(1))

print("\n--- Mean Combined Threat Profiles by Cluster (with translated names) ---")
if 'cluster_threat_profiles_combined_sorted' in locals():
    cluster_threat_profiles_combined_translated = cluster_threat_profiles_combined_sorted.copy()
    cluster_threat_profiles_combined_translated = translate_threat_codes_in_df_columns(cluster_threat_profiles_combined_translated, threat_code_to_name)
    display(cluster_threat_profiles_combined_translated.round(1))
else:
    print("Combined threat cluster profiles not available for display.")

# --- Translate threat names in index for co-occurring threat pairs ---
print("\n--- Top 10 Most Co-occurring Threat Pairs (Overall - with translated names) ---")
overall_cooccurring_threats_translated = overall_cooccurring_threats.copy()
overall_cooccurring_threats_translated = translate_threat_codes_in_df_index(overall_cooccurring_threats_translated, threat_code_to_name)
display(overall_cooccurring_threats_translated.round(1).head(10))

print("\n--- Top 10 Most Co-occurring Threat Pairs (Marine System - with translated names) ---")
marine_cooccurring_threats_translated = marine_cooccurring_threats.copy()
marine_cooccurring_threats_translated = translate_threat_codes_in_df_index(marine_cooccurring_threats_translated, threat_code_to_name)
display(marine_cooccurring_threats_translated.round(1).head(10))

print("\n--- Top 10 Most Co-occurring Threat Pairs (Freshwater System - with translated names) ---")
freshwater_cooccurring_threats_translated = freshwater_cooccurring_threats.copy()
freshwater_cooccurring_threats_translated = translate_threat_codes_in_df_index(freshwater_cooccurring_threats_translated, threat_code_to_name)
display(freshwater_cooccurring_threats_translated.round(1).head(10))

print("\n--- Top 10 Most Co-occurring Threat Pairs (Threatened Species - with translated names) ---")
threatened_cooccurring_threats_translated = threatened_cooccurring_threats.copy()
threatened_cooccurring_threats_translated = translate_threat_codes_in_df_index(threatened_cooccurring_threats_translated, threat_code_to_name)
display(threatened_cooccurring_threats_translated.round(1).head(10))


--- Overall Threat Prevalence (with translated names) ---


,Species Count,Percentage of Species
5_4_4: Unintentional effects: large scale (species being assessed is not the target)[harvest],1081,85.387046
5_4_3: Unintentional effects: subsistence/small scale (species being assessed is not the target)[harvest],727,57.424961
5_4_1: Intentional use: subsistence/small scale (species being assessed is the target)[harvest],340,26.856240
5_4_2: Intentional use: large scale (species being assessed is the target)[harvest],244,19.273302
1_1: Housing & urban areas,134,10.584518
1_2: Commercial & industrial areas,118,9.320695
11_1: Habitat shifting & alteration,84,6.635071
5_4_5: Persecution/control,33,2.606635
"9_3_2: Soil erosion, sedimentation",31,2.448657
3_2: Mining & quarrying,27,2.132701



--- Threat Percentages by System (with translated names) ---


,5_4_4: Unintentional effects: large scale (species being assessed is not the target)[harvest],5_4_1: Intentional use: subsistence/small scale (species being assessed is the target)[harvest],5_4_3: Unintentional effects: subsistence/small scale (species being assessed is not the target)[harvest],5_4_2: Intentional use: large scale (species being assessed is the target)[harvest],9_2_1: Oil spills,1_2: Commercial & industrial areas,11_2: Droughts,4_3: Shipping lanes,5_4_5: Persecution/control,7_2_9: Small dams,...,7_2_4: Abstraction of surface water (unknown use),8_2_2: Named species,5_3_4: Unintentional effects: large scale (species being assessed is not the target)[harvest],"6_2: War, civil unrest & military exercises",7_2_5: Abstraction of ground water (domestic use),7_2_7: Abstraction of ground water (agricultural use),7_1_1: Increase in fire frequency/intensity,2_1_2: Small-holder farming,8_1_1: Unspecified species,8_4_2: Named species
systems,,,,,,,,,,,,,,,,,,,,,
brackish,100.000000,58.333333,100.000000,33.333333,0.000000,33.333333,0.000000,0.000000,8.333333,8.333333,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
freshwater,20.000000,66.666667,75.555556,20.000000,4.444444,17.777778,24.444444,11.111111,35.555556,17.777778,...,2.222222,0.000000,4.444444,2.222222,2.222222,2.222222,6.666667,4.444444,2.222222,0.000000
marine,87.675765,25.062035,56.327543,19.106700,0.909843,8.767577,0.000000,0.165426,1.323408,0.000000,...,0.000000,0.165426,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.082713



--- Threat Percentages by IUCN Red List Category (with translated names) ---


...

,Co-occurrence Count
"(5_4_3: Unintentional effects: subsistence/small scale (species being assessed is not the target)[harvest], 5_4_4: Unintentional effects: large scale (species being assessed is not the target)[harvest])",646
"(5_4_1: Intentional use: subsistence/small scale (species being assessed is the target)[harvest], 5_4_3: Unintentional effects: subsistence/small scale (species being assessed is not the target)[harvest])",290
"(5_4_1: Intentional use: subsistence/small scale (species being assessed is the target)[harvest], 5_4_4: Unintentional effects: large scale (species being assessed is not the target)[harvest])",281
"(5_4_2: Intentional use: large scale (species being assessed is the target)[harvest], 5_4_4: Unintentional effects: large scale (species being assessed is not the target)[harvest])",223
"(5_4_2: Intentional use: large scale (species being assessed is the target)[harvest], 5_4_3: Unintentional effects: subsistence/small scale (species being assessed is not the target)[harvest])",208
"(5_4_1: Intentional use: subsistence/small scale (species being assessed is the target)[harvest], 5_4_2: Intentional use: large scale (species being assessed is the target)[harvest])",186
"(1_1: Housing & urban areas, 5_4_3: Unintentional effects: subsistence/small scale (species being assessed is not the target)[harvest])",113
"(1_1: Housing & urban areas, 5_4_4: Unintentional effects: large scale (species being assessed is not the target)[harvest])",106
"(1_2: Commercial & industrial areas, 5_4_3: Unintentional effects: subsistence/small scale (species being assessed is not the target)[harvest])",103
"(1_2: Commercial & industrial areas, 5_4_4: Unintentional effects: large scale (species being assessed is not the target)[harvest])",96



--- Top 10 Most Co-occurring Threat Pairs (Freshwater System - with translated names) ---


,Co-occurrence Count
"(5_4_1: Intentional use: subsistence/small scale (species being assessed is the target)[harvest], 5_4_3: Unintentional effects: subsistence/small scale (species being assessed is not the target)[harvest])",24
"(5_4_3: Unintentional effects: subsistence/small scale (species being assessed is not the target)[harvest], 9_3_2: Soil erosion, sedimentation)",19
"(5_4_1: Intentional use: subsistence/small scale (species being assessed is the target)[harvest], 9_3_2: Soil erosion, sedimentation)",15
"(3_2: Mining & quarrying, 5_4_1: Intentional use: subsistence/small scale (species being assessed is the target)[harvest])",14
"(5_4_3: Unintentional effects: subsistence/small scale (species being assessed is not the target)[harvest], 7_2_11: Dams (size unknown))",14
"(1_1: Housing & urban areas, 5_4_3: Unintentional effects: subsistence/small scale (species being assessed is not the target)[harvest])",13
"(5_4_1: Intentional use: subsistence/small scale (species being assessed is the target)[harvest], 5_4_5: Persecution/control)",13
"(5_4_3: Unintentional effects: subsistence/small scale (species being assessed is not the target)[harvest], 5_4_5: Persecution/control)",13
"(3_2: Mining & quarrying, 5_4_3: Unintentional effects: subsistence/small scale (species being assessed is not the target)[harvest])",12
"(7_2_11: Dams (size unknown), 9_3_2: Soil erosion, sedimentation)",11



--- Top 10 Most Co-occurring Threat Pairs (Threatened Species - with translated names) ---


,Co-occurrence Count
"(5_4_3: Unintentional effects: subsistence/small scale (species being assessed is not the target)[harvest], 5_4_4: Unintentional effects: large scale (species being assessed is not the target)[harvest])",359
"(5_4_1: Intentional use: subsistence/small scale (species being assessed is the target)[harvest], 5_4_3: Unintentional effects: subsistence/small scale (species being assessed is not the target)[harvest])",230
"(5_4_1: Intentional use: subsistence/small scale (species being assessed is the target)[harvest], 5_4_4: Unintentional effects: large scale (species being assessed is not the target)[harvest])",211
"(5_4_2: Intentional use: large scale (species being assessed is the target)[harvest], 5_4_4: Unintentional effects: large scale (species being assessed is not the target)[harvest])",166
"(5_4_2: Intentional use: large scale (species being assessed is the target)[harvest], 5_4_3: Unintentional effects: subsistence/small scale (species being assessed is not the target)[harvest])",165
"(5_4_1: Intentional use: subsistence/small scale (species being assessed is the target)[harvest], 5_4_2: Intentional use: large scale (species being assessed is the target)[harvest])",151
"(1_1: Housing & urban areas, 5_4_3: Unintentional effects: subsistence/small scale (species being assessed is not the target)[harvest])",107
"(1_2: Commercial & industrial areas, 5_4_3: Unintentional effects: subsistence/small scale (species being assessed is not the target)[harvest])",97
"(1_1: Housing & urban areas, 5_4_4: Unintentional effects: large scale (species being assessed is not the target)[harvest])",97
"(1_2: Commercial & industrial areas, 5_4_4: Unintentional effects: large scale (species being assessed is not the target)[harvest])",88


	6.	Plots + exports
	•	heatmaps, saved figures/tables

In [15]:
import gspread
from gspread_dataframe import set_with_dataframe
from google.colab import auth
from datetime import datetime
import pandas as pd # Added pandas import
import google.auth # Import google.auth

# Authenticate Google Colab for Google Sheets access
auth.authenticate_user()
creds, _ = google.auth.default() # Get default credentials from authenticated user
gc = gspread.Client(creds) # Use gspread.Client with obtained credentials

# Define the name of your Google Sheet for outputs
gsheet_name = 'aaaaResitAiThreat_outputs'

# Global variables to track tables and figure data
global_table_counter = 0
saved_table_info = []

# Helper function to update the INDEX tab
def update_index_tab(worksheet, table_info_list):
    index_df = pd.DataFrame(table_info_list)
    index_df = index_df.sort_values(by='Tab Name').reset_index(drop=True)

    # Clear existing content and write new index
    worksheet.clear()
    set_with_dataframe(worksheet, index_df, row=1, col=1, include_index=False, resize=True)
    print("INDEX tab updated successfully.")


def save_dataframe_to_gsheet(df_to_save, short_name, description):
    global global_table_counter
    global saved_table_info

    global_table_counter += 1
    tab_name = f"Table_{global_table_counter:02d}_{short_name}"

    try:
        # Open the spreadsheet (or create if it doesn't exist)
        try:
            spreadsheet = gc.open(gsheet_name)
        except gspread.exceptions.SpreadsheetNotFound:
            print(f"Creating new Google Sheet: {gsheet_name}")
            spreadsheet = gc.create(gsheet_name)
            # For user authentication, gspread.oauth() usually handles permissions,
            # but explicitly sharing can be done if needed, though user_email might need to be obtained differently.
            # For now, relying on oauth flow for permissions.
            print("Please ensure you grant permissions when prompted.")

        # Check if tab exists, clear or add it
        try:
            worksheet = spreadsheet.worksheet(tab_name)
            # Clear existing content
            worksheet.clear()
            print(f"Cleared existing tab: {tab_name}")
        except gspread.exceptions.WorksheetNotFound:
            worksheet = spreadsheet.add_worksheet(title=tab_name, rows="1", cols="1")
            print(f"Created new tab: {tab_name}")

        # Write DataFrame to the worksheet
        set_with_dataframe(worksheet, df_to_save, row=1, col=1, include_index=False, resize=True)
        print(f"DataFrame saved to Google Sheet tab: {tab_name}")

        # Update saved_table_info list
        saved_table_info.append({
            'Tab Name': tab_name,
            'Description': description,
            'Row Count': len(df_to_save),
            'Last Updated': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        })

        # Ensure INDEX tab exists and update it
        try:
            index_worksheet = spreadsheet.worksheet('INDEX')
        except gspread.exceptions.WorksheetNotFound:
            index_worksheet = spreadsheet.add_worksheet(title='INDEX', rows="1", cols="1")
            print("Created new INDEX tab.")
        update_index_tab(index_worksheet, saved_table_info)

    except Exception as e:
        print(f"Error saving DataFrame to Google Sheet: {e}")
        print("Please ensure correct permissions are granted to the Google Sheet. You might need to manually share the sheet with the account used for authentication.")

print("Google Sheets helper functions defined and authenticated.")

Google Sheets helper functions defined and authenticated.


In [16]:
# Overall Co-occurrence Heatmap for Level 1 Threats
create_cooccurrence_matrix_heatmap(
    threats_level1_dummies,
    threat_code_to_name,
    'Overall Co-occurrence of Level 1 Threats (% of All Species)',
    'overall_level1_cooccurrence'
)

Figure saved to: /content/drive/MyDrive/aaaaResitAiThreat/RawData/Figures/fig_01_overall_level1_cooccurrence.png


In [17]:
# Overall Co-occurrence Heatmap for Combined Threats
if 'X_filtered_combined' in locals() and not X_filtered_combined.empty:
    create_cooccurrence_matrix_heatmap(
        X_filtered_combined,
        threat_code_to_name,
        'Overall Co-occurrence of Combined Filtered Threats (% of All Species)',
        'overall_combined_cooccurrence'
    )
else:
    print("Skipping Combined Threats heatmap: X_filtered_combined is not available or is empty.")

Figure saved to: /content/drive/MyDrive/aaaaResitAiThreat/RawData/Figures/fig_02_overall_combined_cooccurrence.png


In [18]:
# CANONICAL: keep
marine_threat_matrix_level1 = threats_level1_dummies[df['systems'] == 'marine']
create_cooccurrence_matrix_heatmap(
    marine_threat_matrix_level1,
    threat_code_to_name,
    'Marine System Co-occurrence of Level 1 Threats (% of Marine Species)',
    'marine_level1_cooccurrence'
)

Figure saved to: /content/drive/MyDrive/aaaaResitAiThreat/RawData/Figures/fig_03_marine_level1_cooccurrence.png


In [19]:
# CANONICAL: keep
freshwater_threat_matrix_level1 = threats_level1_dummies[df['systems'] == 'freshwater']
create_cooccurrence_matrix_heatmap(
    freshwater_threat_matrix_level1,
    threat_code_to_name,
    'Freshwater System Co-occurrence of Level 1 Threats (% of Freshwater Species)',
    'freshwater_level1_cooccurrence'
)

Figure saved to: /content/drive/MyDrive/aaaaResitAiThreat/RawData/Figures/fig_04_freshwater_level1_cooccurrence.png


In [ ]:
# CANONICAL: keep
threatened_threat_matrix_level1 = threats_level1_dummies[df['threatened'] == True]
create_cooccurrence_matrix_heatmap(
    threatened_threat_matrix_level1,
    threat_code_to_name,
    'Threatened Species Co-occurrence of Level 1 Threats (% of Threatened Species)',
    'threatened_level1_cooccurrence'
)

In [20]:
# Heatmap of Level 1 Cluster Profiles
plt.figure(figsize=(12, 8))
sns.heatmap(
    cluster_threat_profiles_level1_sorted,
    annot=True,
    fmt=".1f",
    cmap="viridis",
    linewidths=.5,
    cbar_kws={'label': 'Mean Percentage of Species Affected'}
)
plt.title('Level 1 Threat Cluster Profiles')
plt.xlabel('Threat')
plt.ylabel('Threat Cluster')
plt.xticks(rotation=90, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
save_figure_to_drive(plt, 'level1_cluster_profile_heatmap')

Figure saved to: /content/drive/MyDrive/aaaaResitAiThreat/RawData/Figures/fig_05_level1_cluster_profile_heatmap.png


In [24]:
print("\n--- Summary of Level 1 Threat Cluster Profiles (Sorted by Threat Code) ---")
display(cluster_threat_profiles_level1_sorted)


--- Summary of Level 1 Threat Cluster Profiles (Sorted by Threat Code) ---


,1,2,3,4,5,6,7,8,9,11
threat_cluster_level1,,,,,,,,,,
0,13.058419,4.810997,3.092784,0.773196,99.656357,2.147766,5.154639,0.515464,5.841924,8.419244
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.000000,100.000000,0.000000,0.000000,0.000000,0.000000,100.000000,100.000000,0.000000,0.000000
3,100.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [22]:
print("\n--- Summary of Level 1 Threat Cluster Attributes (with translated names) ---")
if 'cluster_threat_profiles_level1_sorted' in locals():
    cluster_threat_profiles_level1_translated = cluster_threat_profiles_level1_sorted.copy()
    cluster_threat_profiles_level1_translated = translate_threat_codes_in_df_columns(cluster_threat_profiles_level1_translated, threat_code_to_name)
    display(cluster_threat_profiles_level1_translated.round(1))
else:
    print("Level 1 threat cluster profiles not available for display.")


--- Summary of Level 1 Threat Cluster Attributes (with translated names) ---


,1: Residential & commercial development,2: Agriculture & aquaculture,3: Energy production & mining,4: Transportation & service corridors,5: Biological resource use,6: Human intrusions & disturbance,7: Natural system modifications,"8: Invasive & other problematic species, genes & diseases",9: Pollution,11: Climate change & severe weather
threat_cluster_level1,,,,,,,,,,
0,13.1,4.8,3.1,0.8,99.7,2.1,5.2,0.5,5.8,8.4
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,100.0,0.0,0.0,0.0,0.0,100.0,100.0,0.0,0.0
3,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [23]:
print("\n--- Summary of Combined Threat Cluster Attributes (with translated names) ---")
if 'cluster_threat_profiles_combined_sorted' in locals():
    cluster_threat_profiles_combined_translated = cluster_threat_profiles_combined_sorted.copy()
    cluster_threat_profiles_combined_translated = translate_threat_codes_in_df_columns(cluster_threat_profiles_combined_translated, threat_code_to_name)
    display(cluster_threat_profiles_combined_translated.round(1))
else:
    print("Combined threat cluster profiles not available for display.")


--- Summary of Combined Threat Cluster Attributes (with translated names) ---


,1_1: Housing & urban areas,1_2: Commercial & industrial areas,1_3: Tourism & recreation areas,2_4_1: Subsistence/artisanal aquaculture,2_4_2: Industrial aquaculture,3_1: Oil & gas drilling,3_2: Mining & quarrying,5_4_1: Intentional use: subsistence/small scale (species being assessed is the target)[harvest],5_4_2: Intentional use: large scale (species being assessed is the target)[harvest],5_4_3: Unintentional effects: subsistence/small scale (species being assessed is not the target)[harvest],...,9_1_2: Run-off,9_1_3: Type Unknown/Unrecorded,9_2_1: Oil spills,9_2_2: Seepage from mining,9_3_1: Nutrient loads,"9_3_2: Soil erosion, sedimentation",9_3_3: Herbicides & pesticides,11_1: Habitat shifting & alteration,11_2: Droughts,11_3: Temperature extremes
threat_cluster_combined,,,,,,,,,,,,,,,,,,,,,
0,11.0,9.7,1.2,2.2,2.0,1.0,1.0,28.0,20.5,62.7,...,0.3,0.9,1.0,0.7,0.4,1.1,0.4,7.0,0.0,1.1
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,28.6,14.3,0.0
2,14.3,0.0,14.3,0.0,0.0,0.0,85.7,57.1,0.0,0.0,...,0.0,0.0,0.0,28.6,0.0,14.3,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,50.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,50.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,50.0,0.0,50.0,0.0,0.0,0.0
6,34.8,30.4,34.8,4.3,0.0,0.0,39.1,91.3,26.1,91.3,...,52.2,0.0,4.3,26.1,43.5,78.3,34.8,13.0,43.5,17.4
7,0.0,0.0,0.0,0.0,0.0,100.0,100.0,0.0,100.0,100.0,...,0.0,0.0,100.0,100.0,0.0,0.0,100.0,0.0,0.0,0.0
